In [1]:
%pip install pandas sidrapy
import pandas as pd
from sidrapy import get_table
import unidecode

MES_REF_IPCA = 12

def tratar_nomes_colunas(df: pd.DataFrame):
    df.columns = [unidecode.unidecode(col) for col in df.columns]
    df.columns = [col.lower() for col in df.columns]
    df.columns = [col.replace(" ", "_") for col in df.columns]
    df.columns = [col.replace("-", "_") for col in df.columns]
    df.columns = [col.replace("(", "") for col in df.columns]
    df.columns = [col.replace(")", "") for col in df.columns]
    df.columns = [col.replace(".", "") for col in df.columns]
    df.columns = [col.replace(",", "") for col in df.columns]
    df.columns = [col.replace("'", "") for col in df.columns]
    df.columns = [col.replace("'", "") for col in df.columns]
    return df


def deflacionar_pib(
    df_pib: pd.DataFrame,
    ano_base: int,
    ano_inicio_serie: int,
    ano_fim_serie: int,
    coluna_pib: str,
):
    # dados do IPCA
    ipca_data = get_table(
        table_code="1737",
        territorial_level="1",
        ibge_territorial_code="all",
        variable="69",
        period="all",
        header="y",
    )
    # tratamentos
    ipca_data.columns = ipca_data.iloc[0]
    ipca_data = ipca_data.drop(ipca_data.index[0])
    ipca_data = tratar_nomes_colunas(ipca_data)
    ipca_data["valor"] = ipca_data["valor"].str.replace("..", "0").astype(float)
    ipca_data["ano"] = ipca_data["mes_codigo"].astype(str).str[:4].astype(int)
    ipca_data["mes"] = ipca_data["mes_codigo"].astype(str).str[-2:].astype(int)
    # filtro
    ipca_data = ipca_data.loc[
        (ipca_data["mes"] == MES_REF_IPCA)
        & (ipca_data["ano"] >= ano_inicio_serie)
        & (ipca_data["ano"] <= ano_fim_serie)
    ].copy()
    # deflacionamento
    ipca_data["ipca_acumulado"] = (1 + ipca_data["valor"] / 100).cumprod()
    indice_base = ipca_data[ipca_data["ano"] == ano_base]["ipca_acumulado"].values[0]
    ipca_data["ipca_normalizado"] = ipca_data["ipca_acumulado"] / indice_base
    # aplicação do deflacionamento
    df_merged = pd.merge(
        df_pib, ipca_data[["ano", "ipca_normalizado"]], on="ano", how="left"
    )
    df_merged["pib_deflacionado"] = (
        df_merged[coluna_pib] / df_merged["ipca_normalizado"]
    )
    return df_merged


def calcular_pib_per_capita(codigo_municipio, pop_data):
    # DADOS DO PIB
    # tabela 5938: PIB dos Municípios - Contas Regionais
    pib_osasco = get_table(
        table_code="5938",
        territorial_level="6",  # 6 para municípios
        ibge_territorial_code=codigo_municipio,  #"3534401",  # Código de osasco
        period="all",  # Todos os períodos disponíveis
        variable="all",
        header="y",
    )
    # tratamento do nome das colunas
    pib_osasco.columns = pib_osasco.iloc[0]
    pib_osasco = pib_osasco.drop(pib_osasco.index[0])
    pib_osasco = tratar_nomes_colunas(pib_osasco)
    pib_osasco = pib_osasco.rename(columns={"valor": "pib_corrente"})

    pib_osasco['pib_corrente'] = pib_osasco['pib_corrente'].replace("...", 0)


    # tipo dos dados
    pib_osasco = pib_osasco.astype(
        {
            "pib_corrente": float,
            "ano": int,
            "municipio_codigo": str,
        }
    )
    # pib_osasco['pib_corrente'] = pib_osasco['pib_corrente'] * 1_000
    
    # filtro para pib em valores correntes
    pib_osasco_corrente = pib_osasco.loc[pib_osasco["variavel_codigo"] == "37"].copy()
    # deflacionamento
    pib_osasco_real = deflacionar_pib(
        pib_osasco_corrente,
        ano_base=2023,
        ano_inicio_serie=2002,
        ano_fim_serie=2023,
        coluna_pib="pib_corrente",
    )

    # DADOS DA POPULAÇÃO
    pop_data = pop_data.rename(columns={"valor": "populacao"})
    pop_data = pop_data.astype(
        {
            "populacao": int,
            "ano": int,
            "municipio_codigo": str,
        }
    )
    pop_data = pop_data[["municipio_codigo", "ano", "populacao"]].copy()

    # PIB PER CAPITA
    pib_osasco_real = pd.merge(
        pib_osasco_real, pop_data, on=["municipio_codigo", "ano"], how="left"
    )
    pib_osasco_real["pib_per_capita"] = (
        pib_osasco_real["pib_deflacionado"] * 1_000
    ) / pib_osasco_real["populacao"]

    return pib_osasco_real


def calcular_pib_por_categoria(codigo_municipio):
    # DADOS DO PIB
    # tabela 5938: PIB dos Municípios - Contas Regionais
    pib_osasco = get_table(
        table_code="5938",
        territorial_level="6",  # 6 para municípios
        ibge_territorial_code=codigo_municipio, #"3534401",  # Código de osasco
        period="all",  # Todos os períodos disponíveis
        variable="all",
        header="y",
    )
    # tratamento do nome das colunas
    pib_osasco.columns = pib_osasco.iloc[0]
    pib_osasco = pib_osasco.drop(pib_osasco.index[0])
    pib_osasco = tratar_nomes_colunas(pib_osasco)
    pib_osasco = pib_osasco.rename(columns={"valor": "pib_corrente"})
    # tipo dos dados
    pib_osasco['pib_corrente'] = pib_osasco['pib_corrente'].replace("...", 0)
    pib_osasco = pib_osasco.astype(
        {
            "pib_corrente": float,
            "ano": int,
            "municipio_codigo": str,
        }
    )
    # deflacionamento
    pib_osasco_real = deflacionar_pib(
        pib_osasco,
        ano_base=2023,
        ano_inicio_serie=2002,
        ano_fim_serie=2023,
        coluna_pib="pib_corrente",
    )
    # PIB POR CATEGORIA
    pib_osasco_real["variavel_codigo"] = pib_osasco_real["variavel_codigo"].astype(str)
    pib_osasco_categorias = pib_osasco_real.loc[
        (
            pib_osasco_real["variavel_codigo"].isin(
                ["37", "543", "513", "517", "6575", "525"]
            )
        )
    ].copy()
    categorias = {
        "Produto Interno Bruto a preços correntes": "Total",
        "Impostos, líquidos de subsídios, sobre produtos a preços correntes": "Impostos",
        "Valor adicionado bruto a preços correntes da agropecuária": "Agropecuária",
        "Valor adicionado bruto a preços correntes da indústria": "Indústria",
        "Valor adicionado bruto a preços correntes dos serviços, exclusive administração, defesa, educação e saúde públicas e seguridade social": "Serviços",
        "Valor adicionado bruto a preços correntes da administração, defesa, educação e saúde públicas e seguridade social": "Administração",
    }
    pib_osasco_categorias["variavel_dash"] = pib_osasco_categorias["variavel"].map(
        categorias
    )
    return pib_osasco_categorias


def calcular_participacao_pib_estadual(codigo_municipio):
    pib_sp = get_table(
        table_code="5938",
        territorial_level="3",
        ibge_territorial_code="35",
        period="all",
        variable="37",
        header="y",
    )
    pib_sp.columns = pib_sp.iloc[0]
    pib_sp = pib_sp.drop(pib_sp.index[0])
    pib_sp = tratar_nomes_colunas(pib_sp)
    pib_sp = pib_sp.rename(columns={"valor": "pib_corrente"})
    pib_sp = pib_sp[["ano", "pib_corrente"]].copy()
    pib_sp['pib_corrente'] = pib_sp['pib_corrente'].replace("...", 0)
    pib_sp = pib_sp.astype({"ano": int, "pib_corrente": float})
    # deflacionamento
    pib_sp_real = deflacionar_pib(
        pib_sp,
        ano_base=2023,
        ano_inicio_serie=2002,
        ano_fim_serie=2023,
        coluna_pib="pib_corrente",
    )
    pib_sp_real = pib_sp_real.drop("pib_corrente", axis=1)
    # DADOS DO PIB DE OSASCO
    pib_osasco = calcular_pib_por_categoria(codigo_municipio)
    pib_osasco = pib_osasco.loc[pib_osasco["variavel_dash"] == "Total"].copy()
    pib_osasco = pib_osasco[["ano", "municipio_codigo", "municipio", "pib_deflacionado"]].copy()
    pib_osasco["ano"] = pib_osasco["ano"].astype(int)
    pib_osasco_sp = pib_osasco.merge(
        pib_sp_real, on="ano", how="left", suffixes=("_osasco", "_sp")
    )
    pib_osasco_sp["participacao_pib_sp"] = (
        pib_osasco_sp["pib_deflacionado_osasco"] / pib_osasco_sp["pib_deflacionado_sp"]
    )
    return pib_osasco_sp

StatementMeta(, 8ddf5533-2a5c-46ba-95bd-8e3f5e818b4a, 8, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [19]:
pop_data = spark.sql("SELECT * FROM lh_cidade_inteligente_osasco.gold_osasco_populacao_ibge").toPandas()
pop_data = pop_data.rename(columns={"valor": "populacao"}).copy()
pop_data = pop_data.astype(
    {
        "populacao": int,
        "ano": int,
        "municipio_codigo": str,
    }
)
pop_data = pop_data[["municipio_codigo", "ano", "populacao"]].copy()

anos = pd.DataFrame({"ano": range(pop_data["ano"].min(), 2024 + 1)})

municipios = pop_data[["municipio_codigo"]].drop_duplicates()

pop_data = (
    municipios
    .merge(anos, how="cross")
    .merge(pop_data, on=["municipio_codigo", "ano"], how="left")
    .sort_values(["municipio_codigo", "ano"])
)

pop_data["populacao"] = pop_data.groupby("municipio_codigo")["populacao"].ffill()

# Osasco, Sorocaba, Ribeirão Preto, São Bernardo do Campo, São José dos Campos, Santo André
codigo_municipios = ["3534401", "3552205", "3543402", "3548708", "3549904", "3547809"]
dict_dfs_pib_municipios = {}
dict_dfs_pib_categorias = {}
dict_dfs_participacao_pib_sp = {}

for municipio in codigo_municipios:
    dict_dfs_pib_municipios[municipio] = calcular_pib_per_capita(municipio, pop_data)
    dict_dfs_pib_categorias[municipio] = calcular_pib_por_categoria(municipio)
    dict_dfs_participacao_pib_sp[municipio] = calcular_participacao_pib_estadual(municipio)

pib_per_capita = pd.concat(dict_dfs_pib_municipios.values(), ignore_index=True)
pib_categoria = pd.concat(dict_dfs_pib_categorias.values(), ignore_index=True)
participacao_pib = pd.concat(dict_dfs_participacao_pib_sp.values(), ignore_index=True)

StatementMeta(, 8ddf5533-2a5c-46ba-95bd-8e3f5e818b4a, 27, Finished, Available, Finished, False)

In [24]:
pib_per_capita.query("ano == 2021")['pib_deflacionado'] 

StatementMeta(, 8ddf5533-2a5c-46ba-95bd-8e3f5e818b4a, 32, Finished, Available, Finished, False)

19     9.530579e+10
41     4.928845e+10
63     4.422153e+10
85     6.449954e+10
107    5.003598e+10
129    3.610333e+10
Name: pib_deflacionado, dtype: float64

# Write no lakehouse em Spark

In [27]:
pib_per_capita['populacao'] = pib_per_capita['populacao'].astype(int)

StatementMeta(, 8ddf5533-2a5c-46ba-95bd-8e3f5e818b4a, 35, Finished, Available, Finished, False)

In [29]:
spark_pib_per_capita = spark.createDataFrame(pib_per_capita)
spark_pib_per_capita.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_osasco_pib_per_capita")


spark_pib_categoria = spark.createDataFrame(pib_categoria)
spark_pib_categoria.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_osasco_pib_categoria")


spark_participacao_pib = spark.createDataFrame(participacao_pib)
spark_participacao_pib.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_osasco_participacao_pib")

StatementMeta(, 8ddf5533-2a5c-46ba-95bd-8e3f5e818b4a, 37, Finished, Available, Finished, False)